<a href="https://colab.research.google.com/github/YOUR-USERNAME/bags-vectors-transformers/blob/main/day2/notebooks/4_bonus_embeddings_classifier_solutions.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Bags, Vectors & Transformers
## Day 2 — Embeddings as Features for Classification *(Bonus)*  ·  **SOLUTIONS**

**A Methods Workshop in Computational Text Analysis**
Denise J. Roth · Strategic Communication Group · Wageningen University & Research

---

This is the **solutions** version. Every **✏️ Exercise** is filled in with one possible answer
plus a short comment. Your exact numbers will depend on the sample, but the patterns should hold.


## 0. Setup

In [ ]:
!pip install datasets gensim -q

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

import gensim.downloader as api

print("Setup complete!")

## 1. Loading the policy data

In [ ]:
from datasets import load_dataset

bills = load_dataset("dreamproit/bill_labels_us", split="train")
df = bills.to_pandas()

print("Total bills:", len(df))
print("Columns:", list(df.columns))
df[["title", "policy_area"]].head()

In [ ]:
df = df.rename(columns={"title": "text"})
df = df[["text", "policy_area"]].dropna()

top_areas = df["policy_area"].value_counts().head(6).index.tolist()
df = df[df["policy_area"].isin(top_areas)]

df = (df.groupby("policy_area", group_keys=False)
        .apply(lambda g: g.sample(min(len(g), 800), random_state=42))
        .sample(frac=1, random_state=42)
        .reset_index(drop=True))

print("Policy areas kept:")
print(df["policy_area"].value_counts())

In [ ]:
for area in df["policy_area"].unique()[:5]:
    example = df[df["policy_area"] == area]["text"].iloc[0]
    print(f"[{area}]")
    print(f"   {example}\n")

In [ ]:
train_df, test_df = train_test_split(
    df, test_size=0.25, random_state=42, stratify=df["policy_area"]
)
train_df = train_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)
print("Train:", len(train_df), "| Test:", len(test_df))

In [ ]:
glove = api.load("glove-wiki-gigaword-50")
print("GloVe loaded. Vocab size:", len(glove), "| dim:", glove.vector_size)

## 2. A light clean

In [ ]:
import re

def clean(text):
    text = str(text).lower()
    text = re.sub(r"[^a-z\s]", " ", text)
    return text.split()

train_df["tokens"] = train_df["text"].apply(clean)
test_df["tokens"] = test_df["text"].apply(clean)

y_train = train_df["policy_area"].values
y_test = test_df["policy_area"].values

print("Example tokens:", train_df["tokens"].iloc[0][:12])

## 3. Baseline: TF-IDF features

In [ ]:
train_text = train_df["tokens"].apply(" ".join)
test_text = test_df["tokens"].apply(" ".join)

tfidf = TfidfVectorizer(max_features=5000)
X_train_tfidf = tfidf.fit_transform(train_text)
X_test_tfidf = tfidf.transform(test_text)

clf_tfidf = LogisticRegression(max_iter=1000)
clf_tfidf.fit(X_train_tfidf, y_train)
pred_tfidf = clf_tfidf.predict(X_test_tfidf)

acc_tfidf = accuracy_score(y_test, pred_tfidf)
print(f"TF-IDF baseline accuracy: {acc_tfidf:.3f}")

## 4. Embedding features: averaging word vectors

In [ ]:
def document_vector(tokens, model=glove):
    vectors = [model[t] for t in tokens if t in model]
    if len(vectors) == 0:
        return np.zeros(model.vector_size)
    return np.mean(vectors, axis=0)

X_train_emb = np.vstack([document_vector(toks) for toks in train_df["tokens"]])
X_test_emb = np.vstack([document_vector(toks) for toks in test_df["tokens"]])

print("Embedding feature matrix shape:", X_train_emb.shape)

In [ ]:
clf_emb = LogisticRegression(max_iter=1000)
clf_emb.fit(X_train_emb, y_train)
pred_emb = clf_emb.predict(X_test_emb)

acc_emb = accuracy_score(y_test, pred_emb)
print(f"Embedding-feature accuracy: {acc_emb:.3f}")

## 5. Head-to-head comparison

In [ ]:
print(f"TF-IDF baseline:      {acc_tfidf:.3f}")
print(f"Embedding features:   {acc_emb:.3f}")
diff = acc_emb - acc_tfidf
print(("Embeddings won by " if diff > 0 else "TF-IDF won by ") + f"{abs(diff):.3f}")

In [ ]:
plt.figure(figsize=(6, 5))
plt.bar(["TF-IDF\n(baseline)", "Embeddings\n(averaged)"],
        [acc_tfidf, acc_emb], color=["#1A1A2E", "#34B233"])
plt.ylabel("Test accuracy")
plt.title("TF-IDF vs. embedding features")
plt.ylim(0, 1)
for i, v in enumerate([acc_tfidf, acc_emb]):
    plt.text(i, v + 0.02, f"{v:.3f}", ha="center", fontweight="bold")
plt.tight_layout()
plt.show()

> **✏️ Exercise 1**
>
> Which representation won, and by how much? Does the outcome make sense for short,
> keyword-heavy bill titles?


In [ ]:
# ✅ Solution
print(f"TF-IDF:     {acc_tfidf:.3f}")
print(f"Embeddings: {acc_emb:.3f}")

# Comment: on this task TF-IDF is typically very competitive and often wins outright.
# That makes sense: bill titles are short and dominated by strong topical keywords
# ("tax", "health", "defense", "school"). TF-IDF keys on exactly those words, while
# averaging embeddings dilutes them into a single blurred vector. This is precisely
# the lecture's point — embeddings are an option, not an automatic upgrade, and the
# best representation depends on the task.

## 6. Improving the embedding features: TF-IDF weighting

In [ ]:
idf = dict(zip(tfidf.get_feature_names_out(), tfidf.idf_))

def weighted_document_vector(tokens, model=glove, idf=idf):
    vectors, weights = [], []
    for t in tokens:
        if t in model and t in idf:
            vectors.append(model[t])
            weights.append(idf[t])
    if not vectors:
        return np.zeros(model.vector_size)
    vectors = np.array(vectors)
    weights = np.array(weights).reshape(-1, 1)
    return (vectors * weights).sum(axis=0) / weights.sum()

X_train_w = np.vstack([weighted_document_vector(toks) for toks in train_df["tokens"]])
X_test_w = np.vstack([weighted_document_vector(toks) for toks in test_df["tokens"]])

clf_w = LogisticRegression(max_iter=1000)
clf_w.fit(X_train_w, y_train)
pred_w = clf_w.predict(X_test_w)
acc_w = accuracy_score(y_test, pred_w)

print(f"Plain average:            {acc_emb:.3f}")
print(f"TF-IDF weighted average:  {acc_w:.3f}")

> **✏️ Exercise 2**
>
> Did TF-IDF weighting help? Add its bar to the comparison chart (three bars).


In [ ]:
# ✅ Solution
labels = ["TF-IDF", "Embeddings\n(plain)", "Embeddings\n(weighted)"]
accs = [acc_tfidf, acc_emb, acc_w]
colors = ["#1A1A2E", "#34B233", "#0F9B8E"]

plt.figure(figsize=(7, 5))
plt.bar(labels, accs, color=colors)
plt.ylabel("Test accuracy")
plt.title("Three representations, same classifier and data")
plt.ylim(0, 1)
for i, v in enumerate(accs):
    plt.text(i, v + 0.02, f"{v:.3f}", ha="center", fontweight="bold")
plt.tight_layout()
plt.show()

# Comment: TF-IDF weighting usually nudges the embedding average UP a little, because
# distinctive policy words ("medicare", "immigration") now count more than filler
# ("act", "bill", "to"). It often narrows the gap with TF-IDF but rarely overtakes it
# on such keyword-driven text. Still a cheap, worthwhile improvement over a plain average.

## 7. Why might embeddings help? A qualitative look

In [ ]:
def cosine(a, b):
    if np.all(a == 0) or np.all(b == 0):
        return 0.0
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

t1 = "a bill to lower prescription drug prices".split()
t2 = "an act to reduce the cost of medicine".split()
t3 = "a bill to modernize the naval fleet".split()

v1, v2, v3 = document_vector(t1), document_vector(t2), document_vector(t3)
print("Same policy area, different words (health):")
print(f"  t1 vs t2: {cosine(v1, v2):.3f}")
print("Different policy area (defense):")
print(f"  t1 vs t3: {cosine(v1, v3):.3f}")

> **✏️ Exercise 3**
>
> Compare that to what TF-IDF would see. Confirm `t1` and `t2` are nearly orthogonal under
> TF-IDF. What does this tell you about when embeddings should win?


In [ ]:
# ✅ Solution
from sklearn.metrics.pairwise import cosine_similarity

titles = [" ".join(t1), " ".join(t2), " ".join(t3)]
tfidf_vecs = tfidf.transform(titles)

sims = cosine_similarity(tfidf_vecs)
print("TF-IDF cosine similarities:")
print(f"  t1 vs t2 (same topic, diff words): {sims[0,1]:.3f}")
print(f"  t1 vs t3 (different topic):        {sims[0,2]:.3f}")

# Comment: under TF-IDF, t1 and t2 are almost ORTHOGONAL (~0) because they share
# essentially no content words — "prescription/drug/prices" vs "cost/medicine".
# TF-IDF cannot see that they mean the same thing. Embeddings scored them clearly
# similar. So embeddings should win when the SAME meaning is expressed with DIFFERENT
# vocabulary — paraphrase-heavy text, synonyms, varied phrasing. On short titles that
# reuse the same keywords, that advantage rarely gets to matter, which is why TF-IDF
# stays competitive here.

## Wrap-up

That's the full solution set. The recurring lesson: **try both representations and let the
held-out data decide.**

### Optional challenge


In [ ]:
# ✅ Solution — Part 1: concatenate TF-IDF and embedding features
from scipy.sparse import hstack, csr_matrix

X_train_combo = hstack([X_train_tfidf, csr_matrix(X_train_emb)])
X_test_combo = hstack([X_test_tfidf, csr_matrix(X_test_emb)])

clf_combo = LogisticRegression(max_iter=1000)
clf_combo.fit(X_train_combo, y_train)
acc_combo = accuracy_score(y_test, clf_combo.predict(X_test_combo))

print(f"TF-IDF only:          {acc_tfidf:.3f}")
print(f"Embeddings only:      {acc_emb:.3f}")
print(f"Combined (both):      {acc_combo:.3f}")

# Comment: concatenating gives the classifier BOTH the sharp keyword signal (TF-IDF)
# and the meaning signal (embeddings). It often matches or slightly beats the better
# of the two alone — a cheap way to hedge.

In [ ]:
# ✅ Solution — Part 2: confusion matrix for the embedding classifier
labels_sorted = sorted(df["policy_area"].unique())
cm = confusion_matrix(y_test, pred_emb, labels=labels_sorted)

fig, ax = plt.subplots(figsize=(8, 7))
im = ax.imshow(cm, cmap="Greens")
ax.set_xticks(range(len(labels_sorted)))
ax.set_yticks(range(len(labels_sorted)))
ax.set_xticklabels(labels_sorted, rotation=45, ha="right", fontsize=8)
ax.set_yticklabels(labels_sorted, fontsize=8)
ax.set_xlabel("Predicted"); ax.set_ylabel("True")
ax.set_title("Confusion matrix — embedding classifier")
for i in range(len(labels_sorted)):
    for j in range(len(labels_sorted)):
        ax.text(j, i, cm[i, j], ha="center", va="center",
                color="white" if cm[i, j] > cm.max()/2 else "black", fontsize=8)
plt.tight_layout()
plt.show()

# Comment: look for off-diagonal clusters — e.g. related policy areas that get mixed
# up. Confusions often make intuitive sense (topically adjacent areas share vocabulary),
# which is a good reminder that classifier errors are frequently meaningful, not random.